# Construção Pipeline RAG de Livros 
**Autor:** Isaac Lima Moreira

### **Objetivo:** 
Desenvolver um Pipeline completo de Recuperação de Informação Vetorizada Baseada em Embeddings.

### **Aprendizado:** 
- Tratamento de dados, 
- Associação a Chunks, 
- Retrieval-Augmented Generation, 
- Geração de Embeddings, 
- Busca Semântica.


### **Abstração:** 
- Configuração do Ambiente Virtual e Importação de Bibliotecas,
- Limpeza de Dados e Criação de Dataframe contendo o conteúdo dos livros,
- Dividir o conteúdo dos dados em chunks de 250 tokens,
- Atribuir para cada chunk um valor matemático, gerando Embeddings LLM,
- Vetorizar os Embeddings e posicionar cada chunk de forma funcional,
- Testar com as perguntas fáceis e difíceis.

---

# 1 Configurando Ambiente e Importando Bibliotecas

### 1.1 Instalando no Ambiente Virtual as Bibliotecas Externas:
- Pandas
- BeautifulSoup
- Lang Chain Text Splitters
- Sentence Transformers
- Scikit Learn
- IPhyton Display

In [38]:
%pip install beautifulsoup4 pandas langchain-text-splitters sentence-transformers scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 1.2 Importando as Bibliotecas Instaladas na Venv para dentro do escopo

- Bibliotecas nativas do Python
- Bibliotecas recém instaladas no Ambiente Virtual

In [39]:
# Bibliotecas nativas do python:
import os
import glob
import pickle

# Biblioteca Manipulação de dados:
from bs4 import BeautifulSoup
import pandas as pd

# Bibliotecas de NLP:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# Bibliotecas de ML:
from sklearn.metrics.pairwise import cosine_similarity

# Bibliotecas de visualização:
from IPython.display import display, Markdown

## Função das Bibliotecas:

**Pandas:** Gerar DataFrames, organizar dados e salvar arquivos modificados.

**Beautiful Soup:** Limpeza de HTML removendo tags e texto bruto desnecessário para o treinamento do modelo.

**Lang Chain Text Splitters:** Separar o texto bruto do HTML pela pontuação e transformar isso em Chunks.

**Sentence Transformers:** Fazer a Vetorização, transformando a linguagem humana em linguagem computacional utilizando Embeddings.

**SciKit Learn:** Calcular a similaridade dos ângulos dos vetores e gerar a proximidade entre os Chunks.

**iPython Display:** Fazer a formatação de forma organizada.

---


# 2 Limpeza dos Livros HTML utilizando Pandas e BeautifulSoup

Neste bloco será feito a manipulação de dados com o Pandas e BeautifulSoup, visando extrair o conteúdo principal do livro html sem as classes padrão do formato de arquivo.

Para que isso seja possível, optei por criar um Loop 'for' para percorrer cada arquivo no diretório, que foi salvo na variável livros.

### O Loop realizará:

- Extração do título do livro a partir do nome do arquivo
- Leitura do arquivo HTML
- Separação de cada documento dentro de dicionários
- Implementação dos dicionários dentro da lista_livros
- Criação de Dataframe da lista_livros
- Saída dos primeiros caracteres de cada documento para fins de análise

In [40]:
# Abaixo eu estarei definindo o caminho para os arquivos HTML que contêm os resumos dos livros:
caminholivros = 'data/raw/*.html'

livros = glob.glob(caminholivros) # Cria uma lista com os caminhos dos arquivos HTML encontrados

lista_livros = [] # Lista vazia para armazenar todos os livros tratados após o loop for

# Optou-se por utilizar um loop for para iterar cada arquivo, extrair o título e o texto limpo, e armazenar essas informações em uma lista de dicionários. 
# Cada dicionário conterá o título do livro e seu respectivo texto limpo, facilitando a criação de um DataFrame posteriormente.
for livro in livros:
    
    titulo = os.path.basename(livro).split('.')[0] # Extrai o nome do arquivo sem a extensão para usar como título do livro
    
    with open(livro, 'r', encoding='utf-8') as f: # Abre o arquivo HTML para leitura
        livro_html = f.read() # Lê o conteúdo do arquivo HTML e armazena na variável livro_html 
    
    # Criando um objeto BeautifulSoup para analisar o conteúdo HTML:
    soup = BeautifulSoup(livro_html, 'html.parser')
    
    # Utilizarei o separator da biblioteca BeautifulSoup para evitar que as palavras fiquem grudadas após a extração do texto
    texto_limpo = soup.get_text(separator=' ') 
    
    lista_livros.append({ # Adiciona um dicionário com o título e o texto limpo à lista de livros
        'titulo': titulo, 
        'texto': texto_limpo
    })
    
    # Criando um DataFrame a partir da lista de livros:
df_livros = pd.DataFrame(lista_livros)

display(Markdown(f"### Livro: {titulo}")) # Exibe o título do livro como um cabeçalho
    
    # Imprimindo os primeiros 100 caracteres do resumo do livro para verificar se o texto foi extraído corretamente:
display(Markdown(f"### Resumo: {texto_limpo[:500]}..."))

### Livro: The King in Yellow

### Resumo: 
 
 The King in Yellow | Project Gutenberg 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 The Project Gutenberg eBook of  The King in Yellow 
 This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at  www.gutenberg.org . If you are not located in the United States,
you will ...

---

# 3 Segmentacao de Texto (Chunking)

A etapa de Segmentação de Texto por meio de Chunks, consiste em separar os textos completos de cada livro em pedaços menores, facilitando o modelo a separar orações e informações diferentes.

A separação dos textos para cada chunk será feito a partir da Biblioteca LangChain, utilizando o método `RecursiveCharacterTextSplitter`, que utiliza a pontuação como parâmetro base para a criação de Chunks.

O overlap também será criado nessa etapa, a fim de evitar perda de contexto em chunks diferentes.

Por fim, cada chunk receberá metadados contendo sua origem (título do livro) e um ID único, conforme as exigências do projeto.



In [41]:
# configurando o TextSplitter para dividir o texto em chunks menores:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Tamanho máximo de cada chunk, equivalente à 250 tokens
    chunk_overlap=150, # Sobreposição entre os chunks para manter o contexto (overlap), equivalente à um pouco mais de 60 tokens
    
    # Aqui eu estou aplicando como parâmetro para separar os textos, paragrafos, linhas, palavras e caracteres, respectivamente. 
    # O TextSplitter irá tentar dividir o texto usando os separadores na ordem definida, garantindo que os chunks sejam formados de maneira lógica e coerente.
    separators=['\n\n', '\n', ' ', '']
) 

# Criando uma lista vazia para armazenar os chunks de texto:
chunks = []
id_chunk = 0 # Variável contadora para atribuir um ID único a cada chunk

# Utilizando um laço for para aplicar o TextSplitter em cada resumo dos livros:
for index, row in df_livros.iterrows():
    titulo = row['titulo'] # Obtém o título do livro
    texto = row['texto'] # Obtém o texto do resumo do livro
    
    # Aplica o TextSplitter para dividir o texto em chunks menores:
    pedacos = text_splitter.split_text(texto) # Divide o texto em chunks usando o TextSplitter configurado
    
    for pedaco in pedacos: # Itera sobre cada chunk gerado
        chunks.append({
            'id' : f"chunk_{id_chunk}", # Atribui um ID único ao chunk usando a variável contadora 
            'titulo': titulo, # Armazena o título do livro associado ao chunk
            'texto': pedaco # Armazena o texto do chunk
        })
        id_chunk += 1 # Incrementa a variável contadora para o próximo chunk

# Transformando a lista em DataFrame para que os dados possam ser vetorizados nas próximas células:
df_chunks = pd.DataFrame(chunks)

# Exibindo os resultados:
display(Markdown(f"### Chunking Concluído! Total de **{len(df_chunks)} chunks** gerados."))
display(df_chunks.head()) 


### Chunking Concluído! Total de **7213 chunks** gerados.

,id,titulo,texto
0,chunk_0,The Blue Castle,The Blue Castle | Project Gutenberg \n \n \n \...
1,chunk_1,The Blue Castle,Original publication : United States: Frederic...
2,chunk_2,The Blue Castle,CHAPTER XXI \n \n \n CHAPTER XXII \n \n \n ...
3,chunk_3,The Blue Castle,what happened to her because of it.\n \n \nVal...
4,chunk_4,The Blue Castle,Herbert. What hurt her was that she had never ...


---

# 4 Estruturacao e Geração de Embeddings RAG

Nesta etapa, será feita a estruturação de Embedding, transformando a linguagem humana em representações numéricas.

Será utilizado o modelo `all-MiniLM-L6-v2`da biblioteca sentence-transformers, almejando realizar o mapeamento de cada chunk de parágrafos e traduzi-los em um espaço vetorial de 384 dimensões.

Com o método de Embeddings, é possível atribuir um valor semântico próximo através da Similaridade de Cosseno, sendo assim, a máquina associa ou não uma palavra à outra.

In [42]:
# Instanciando o modelo de linguagem (LLM) focado em Embeddings
# Optou-se utilizar o modelo 'all-MiniLM-L6-v2', que é conhecido por ser eficiente e fornecer boas representações vetoriais.
modelo = SentenceTransformer('all-MiniLM-L6-v2')

# Extraindo a coluna de texto dos chunks para criar uma lista de textos que serão vetorizados:
lista_textos = df_chunks['texto'].tolist()

# Gerando a matriz de embeddings para os textos dos chunks usando o modelo instanciado
# O modelo irá transformar cada chunk de texto em um vetor numérico (embedding) que captura o significado semântico do texto.
matriz_embeddings = modelo.encode(lista_textos, show_progress_bar=True) 
# A opção show_progress_bar=True exibe uma barra de progresso durante a vetorização, útil para acompanhar o andamento do processo.

# Adicionando uma nova coluna ao dataframe nomeada 'embedding' para armazenar os embeddings gerados para cada chunk.
# A matriz de embeddings é convertida para uma lista para que possa ser armazenada corretamente na coluna do DataFrame.
df_chunks['embedding'] = matriz_embeddings.tolist()

# Imprimindo resultados:
display(Markdown(f"Cada chunk agora possui uma representação vetorial de **{len(df_chunks['embedding'].iloc[0])} dimensões**."))
display(df_chunks.head(3))

Batches: 100%|██████████| 226/226 [02:19<00:00,  1.62it/s]


Cada chunk agora possui uma representação vetorial de **384 dimensões**.

,id,titulo,texto,embedding
0,chunk_0,The Blue Castle,The Blue Castle | Project Gutenberg \n \n \n \...,"[-0.01642109826207161, -0.02204734832048416, 0..."
1,chunk_1,The Blue Castle,Original publication : United States: Frederic...,"[-0.0013578791404142976, -0.02899017184972763,..."
2,chunk_2,The Blue Castle,CHAPTER XXI \n \n \n CHAPTER XXII \n \n \n ...,"[-0.03655105084180832, 0.024408886209130287, 0..."


---

# 5 Salvamento da Estrutura (Pickle)

Após gerar o Embedding, é uma boa prática de Engenharia de Dados, persistir (salvar) o estado atual do processamento em disco.

Como solicitado, será utilizado o formato binário `pkl`(Pickle), que serializa o objeto Python, mas mantém a estrutura de dados original salva na memória.

In [43]:
# Adicionando um diretório para armazenar os arquivos de embeddings:
diretorio_destino = 'data/processed'
caminho_arquivo = f'{diretorio_destino}/banco_vetorial.pkl' 

# Explicação: Garantimos que o diretório exista antes de salvar.
# O arquivo 'banco_vetorial.pkl' conterá o DataFrame 'df_chunks' com os embeddings gerados.
# Usamos o protocolo mais alto do pickle para eficiência e compatibilidade.
os.makedirs(diretorio_destino, exist_ok=True)

# Salvando o DataFrame de chunks com embeddings em formato binário (pickle):
with open(caminho_arquivo, 'wb') as f: 
    pickle.dump(df_chunks, f, protocol=pickle.HIGHEST_PROTOCOL)

# Confirmação visual: exibe onde o arquivo foi salvo e quantos chunks foram persistidos.
display(Markdown(f"### Arquivo salvo: {caminho_arquivo} — {len(df_chunks)} chunks"))
display(Markdown(f"Tamanho do arquivo gerado: **{os.path.getsize(caminho_arquivo) / (1024 * 1024):.2f} MB**"))
# Explicação: foi utilizado 1024 * 1024 para converter bytes em megabytes,
# pois 1 MB = 1024 bytes * 1024 bytes (base binária de unidades de armazenamento).

### Arquivo salvo: data/processed/banco_vetorial.pkl — 7213 chunks

Tamanho do arquivo gerado: **30.70 MB**

---

# 6 Vetorizacao e Construcao da Base de Dados

Nessa célula, tendo em vista que a base de dados necessária, está persistida em disco e carregada em memória, iniciamos o Retriever.
O Retriever é uma das partes fundamentais na construção de uma Pipeline RAG.

O objetivo desta célula é criar uma função algoritmo capaz de realizar uma busca semântica. Diferente de buscas tradicionais por palavras-chave (onde o sistema apenas procura por termos exatos), a busca semântica entende o contexto e a intenção por trás da pergunta do usuário.

## Funcionamento do Retriever:
- A string enviada pelo usuário é submetida ao mesmo modelo `all-MiniLM-L6-v2`. Isso garante que a pergunta seja projetada no mesmo espaço vetorial de 384 dimensões que os chunks de livros.
- Utlizando a biblioteca `scikit-learn`, calculamos o cosseno do ângulo formado entre o vetor da pergunta e cada um dos vetores da base de dados. O resultado varia de `-1` (totalmente opostos) a `1` (perfeitamente idênticos).
- O algoritmo ordena todos os registros do DataFrame com base nessa pontuação matemática (`score`) e filtra apenas os `top_k` (por padrão, os 3 principais) pedaços de texto semanticamente mais próximos da dúvida do usuário.


In [44]:
# Carrega o DataFrame persistido em disco com o banco vetorial de chunks e embeddings
df_banco = pd.read_pickle(caminho_arquivo) 

def buscar_informacao(pergunta, df_banco, modelo, top_k=3):

    # Gera o embedding da pergunta usando o mesmo modelo usado para os chunks
    vetor_pergunta = modelo.encode([pergunta])
    
    # Converte a coluna de embeddings do DataFrame em uma matriz NumPy
    import numpy as np
    matriz_banco = np.array(df_banco['embedding'].tolist())
    
    # Calcula a similaridade de cosseno entre o embedding da pergunta e os embeddings dos chunks
    similaridades = cosine_similarity(vetor_pergunta, matriz_banco)[0]
    
    # Cria uma cópia do DataFrame e adiciona a coluna de score
    df_resultados = df_banco.copy()
    df_resultados['score'] = similaridades
    
    # Ordena os resultados pelo score em ordem decrescente e retorna os top_k
    df_top_k = df_resultados.sort_values(by='score', ascending=False).head(top_k)
    
    return df_top_k[['titulo', 'score', 'texto']]


display(Markdown("### Motor de Busca Compilado"))
display(Markdown("O banco vetorial foi carregado do disco e a função `buscar_informacao` está pronta."))

### Motor de Busca Compilado

O banco vetorial foi carregado do disco e a função `buscar_informacao` está pronta.

# 7 Testes (Perguntas Faceis e Perguntas Dificeis)

Esta célula pretende validar a eficácia da Pipeline RAG, respondendo as perguntas disponibilizadas pela Compass.

Para isso:
- Criou-se uma lista com as perguntas fáceis e difíceis, separadas por etiqueta.
- Criou-se um laço de repetição 'for' para verificar a etiqueta de cada pergunta e identificar se é difícil ou fácil.
- Dentro do for, está a operação que irá fazer a busca semântica.

In [45]:
display(Markdown("# Iniciando Testes do RAG"))

perguntas_teste = [
    # TESTES FÁCEIS 
    {"nivel": "Fácil", "query": "In The Importance of Being Earnest, what are the names used by Jack when he creates his false identity?"},
    {"nivel": "Fácil", "query": "In The Strange Case of Dr. Jekyll and Mr. Hyde, what is the name of the lawyer who investigates Jekyll’s situation?"},
    {"nivel": "Fácil", "query": "In Alice’s Adventures in Wonderland, what does Alice drink or eat that causes her to change size?"},
    {"nivel": "Fácil", "query": "In Romeo and Juliet, what is the name of Juliet’s nurse?"},
    {"nivel": "Fácil", "query": "In The Great Gatsby, who hosts the parties at the Gatsby mansion?"},
    {"nivel": "Fácil", "query": "In A Room with a View, in which country does Lucy Honeychurch travel during the story?"},
    {"nivel": "Fácil", "query": "In The Blue Castle, what illness does Valancy think she has at the beginning of the story?"},
    {"nivel": "Fácil", "query": "In Frankenstein, where does Victor Frankenstein create the creature?"},
    {"nivel": "Fácil", "query": "In The King in Yellow, what is the name of the forbidden play mentioned in the book?"},
    {"nivel": "Fácil", "query": "In The Picture of Dorian Gray, who paints Dorian’s portrait?"},
    {"nivel": "Fácil", "query": "In The Enchanted April, where do the main characters stay in Italy?"},
    {"nivel": "Fácil", "query": "In The Adventures of Sherlock Holmes, what is Sherlock Holmes’s address?"},
    {"nivel": "Fácil", "query": "In Adventures of Huckleberry Finn, what is the name of the runaway slave Huck travels with?"},
    {"nivel": "Fácil", "query": "In Wuthering Heights, what is the name of the house owned by the Linton family?"},
    {"nivel": "Fácil", "query": "In Pride and Prejudice, how many Bennet sisters are there?"},

    # TESTES DIFÍCEIS 
    {"nivel": "Difícil", "query": "Who has a double life and hides their true identity?"},
    {"nivel": "Difícil", "query": "Who runs away from home and lives an adventure?"},
    {"nivel": "Difícil", "query": "What books in the list explore themes of dual identity or internal conflict?"},
    {"nivel": "Difícil", "query": "Which works present the strongest social critique?"},
    {"nivel": "Difícil", "query": "Which characters in the list undergo significant moral transformation?"},
    {"nivel": "Difícil", "query": "Which works use isolated or symbolic settings to reflect psychological states?"},
    {"nivel": "Difícil", "query": "What is the central reason characters create false identities in The Importance of Being Earnest?"},
    {"nivel": "Difícil", "query": "How does Jekyll’s transformation into Hyde represent the conflict between morality and human instinct?"},
    {"nivel": "Difícil", "query": "What events illustrate the absurd logic of Wonderland in Alice’s Adventures in Wonderland?"},
    {"nivel": "Difícil", "query": "Besides the family feud, what factors lead to the tragedy in Romeo and Juliet?"},
    {"nivel": "Difícil", "query": "How does the green light symbolize dreams and illusion in The Great Gatsby?"},
    {"nivel": "Difícil", "query": "How does Lucy Honeychurch’s perception of freedom change in A Room with a View?"},
    {"nivel": "Difícil", "query": "What motivates Valancy’s transformation in The Blue Castle?"},
    {"nivel": "Difícil", "query": "Who is morally responsible for Frankenstein’s creature’s actions?"},
    {"nivel": "Difícil", "query": "How does the “King in Yellow” function as psychological horror?"},
    {"nivel": "Difícil", "query": "What is the relationship between the portrait and Dorian Gray’s moral decline?"},
    {"nivel": "Difícil", "query": "How does the Italian setting influence transformation in The Enchanted April?"},
    {"nivel": "Difícil", "query": "What deductive methods does Sherlock Holmes use?"},
    {"nivel": "Difícil", "query": "How does Huck’s journey down the river affect his moral growth?"},
    {"nivel": "Difícil", "query": "How does Heathcliff and Catherine’s relationship shape Wuthering Heights?"},
    {"nivel": "Difícil", "query": "How do Elizabeth Bennet’s prejudices affect her perception of Darcy?"}
]

# Exibe o cabeçalho inicial
display(Markdown("## Resultados da Bateria de Testes RAG\n---"))

# 2. Executando o Motor de Busca em loop
for i, teste in enumerate(perguntas_teste, 1):
    nivel = teste["nivel"]
    query = teste["query"]
    
    # Busca apenas o melhor chunk (top_k=1)
    resultado = buscar_informacao(pergunta=query, df_banco=df_banco, modelo=modelo, top_k=1)
    
    # Extrai os dados da linha retornada
    livro = resultado.iloc[0]['titulo']
    score = resultado.iloc[0]['score']
    texto = resultado.iloc[0]['texto']
    
    # Trunca o texto para 1000 caracteres para manter o log limpo e organizado
    texto_curto = texto[:1000].strip() + "..." if len(texto) > 300 else texto
    
    # Imprime de forma minimalista
    display(Markdown(f"**{i}. [{nivel}]** _{query}_"))
    display(Markdown(f"**Livro:** `{livro}` | **Score:** `{score:.4f}`"))
    display(Markdown(f"> {texto_curto}"))
    display(Markdown("<br>")) # Espaçamento entre as perguntas

display(Markdown("---"))
display(Markdown("### Testes finalizados com sucesso!"))

# Iniciando Testes do RAG

## Resultados da Bateria de Testes RAG
---

**1. [Fácil]** _In The Importance of Being Earnest, what are the names used by Jack when he creates his false identity?_

**Livro:** `The Importance of Being Earnest A Trivial Comedy for Serious People` | **Score:** `0.5306`

> LADY BRACKNELL. 
Yes, I remember now that the General was called Ernest, I knew I had some
particular reason for disliking the name.
 
 GWENDOLEN. 
Ernest! My own Ernest! I felt from the first that you could have no other name!
 
 JACK. 
Gwendolen, it is a terrible thing for a man to find out suddenly that all his
life he has been speaking nothing but the truth. Can you forgive me?
 
 GWENDOLEN. 
I can. For I feel that you are sure to change.
 
 JACK. 
My own one!
 
 CHASUBLE. 
[To  Miss Prism .] Lætitia! [Embraces her]
 
 MISS PRISM. 
[Enthusiastically.] Frederick! At last!
 
 ALGERNON. 
Cecily! [Embraces her.] At last!
 
 JACK. 
Gwendolen! [Embraces her.] At last!
 
 LADY BRACKNELL. 
My nephew, you seem to be displaying signs of triviality.
 
 JACK. 
On the contrary, Aunt Augusta, I’ve now realised for the first time in my
life the vital Importance of Being Earnest.
 
 
TABLEAU...

...

**Livro:** `Pride and Prejudice` | **Score:** `0.7630`

> Elizabeth said as little to either as civility would allow, and sat down
again to her work, with an eagerness which it did not often command. She
had ventured only one glance at Darcy. He looked serious as usual; {411}  and,
she thought, more as he had been used to look in Hertfordshire, than as
she had seen him at Pemberley. But, perhaps, he could not in her
mother’s presence be what he was before her uncle and aunt. It was a
painful, but not an improbable, conjecture. 
 Bingley she had likewise seen for an instant, and in that short period
saw him looking both pleased and embarrassed. He was received by Mrs.
Bennet with a degree of civility which made her two daughters ashamed,
especially when contrasted with the cold and ceremonious politeness of
her courtesy and address of his friend. 
 Elizabeth particularly, who knew that her mother owed to the latter the
preservation of her favourite daughter from irremediable infamy, was...

<br>

---

### Testes finalizados com sucesso!

# **Próximos Passos (Trabalhos Futuros):**

Como o foco desta implementação foi manter o padrão de Busca e Recuperação Semântica (Retrieval), o sistema entrega o trecho exato onde a informação reside. O próximo passo evolutivo natural para esta arquitetura seria acoplar a saída deste motor de busca a um modelo generativo, para que ele leia o contexto recuperado e formule respostas com linguagem natural e conversacional.